In [1]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# MultiHeadAttention

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, dropout):
        super().__init__()
        assert emb_dim % num_heads == 0 , "Emb_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        self.wq = nn.Linear(emb_dim, emb_dim)
        self.wk = nn.Linear(emb_dim, emb_dim)
        self.wv = nn.Linear(emb_dim, emb_dim)
        self.proj = nn.Linear(emb_dim, emb_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
      batch_size, Lq, emb_dim = query.shape
      _, Lk, _ = key.shape

      Q = self.wq(query)
      K = self.wk(key)
      V = self.wv(value)

      Q = Q.view(batch_size, Lq, self.num_heads, self.head_dim).transpose(1, 2)
      K = K.view(batch_size, Lk, self.num_heads, self.head_dim).transpose(1, 2)
      V = V.view(batch_size, Lk, self.num_heads, self.head_dim).transpose(1, 2)

      attention_score = Q @ K.transpose(2, 3) / (self.head_dim ** 0.5)

      if attn_mask is not None:
          attention_score = attention_score.masked_fill(attn_mask, -torch.inf)

      if key_padding_mask is not None:
          mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
          attention_score = attention_score.masked_fill(mask, -torch.inf)

      weights = F.softmax(attention_score, dim=-1)
      outputs = self.dropout(weights) @ V

      outputs = outputs.transpose(1, 2).contiguous().view(batch_size, Lq, emb_dim)
      outputs = self.proj(outputs)

      return outputs, weights

# Check
torch.manual_seed(42)
x = torch.randn([2, 3, 8])
mha = MultiHeadAttention(emb_dim=8, num_heads=2, dropout=0.3)
mha_outputs ,_= mha(x,x, x)
assert x.shape == mha_outputs.shape

In [ ]:
# MultiHeadAttention

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, dropout):
        super().__init__()
        assert emb_dim % num_heads == 0 , "Emb_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        self.wq = nn.Linear(emb_dim, emb_dim)
        self.wk = nn.Linear(emb_dim, emb_dim)
        self.wv = nn.Linear(emb_dim, emb_dim)
        self.proj = nn.Linear(emb_dim, emb_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
      batch_size, Lq, emb_dim = query.shape
      _, Lk, _ = key.shape

      Q = self.wq(query)
      K = self.wk(key)
      V = self.wv(value)

      Q = Q.view(batch_size, Lq, self.num_heads, self.head_dim).transpose(1, 2)
      K = K.view(batch_size, Lk, self.num_heads, self.head_dim).transpose(1, 2)
      V = V.view(batch_size, Lk, self.num_heads, self.head_dim).transpose(1, 2)

      attention_score = Q @ K.transpose(2, 3) / (self.head_dim ** 0.5)

      if attn_mask is not None:
          attention_score = attention_score.masked_fill(attn_mask, -torch.inf)

      if key_padding_mask is not None:
          mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
          attention_score = attention_score.masked_fill(mask, -torch.inf)

      weights = F.softmax(attention_score, dim=-1)
      outputs = self.dropout(weights) @ V

      outputs = outputs.transpose(1, 2).contiguous().view(batch_size, Lq, emb_dim)
      outputs = self.proj(outputs)

      return outputs, weights

# Check
torch.manual_seed(42)
x = torch.randn([2, 3, 8])
mha = MultiHeadAttention(emb_dim=8, num_heads=2, dropout=0.3)
mha_outputs ,_= mha(x,x, x)
assert x.shape == mha_outputs.shape

In [3]:
#TransformerEncoderBlock

class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout, ffn_hidden_dim):
        super().__init__()

        # Multi-head self attention
        self.self_attn = MultiHeadAttention(embed_dim, num_heads, dropout)

        # Layer norms
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Feed-forward network
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, embed_dim))

        # Dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        # Multi-head self attention
        attn_output, _ = self.self_attn(x, x, x, attn_mask=attn_mask,
                                        key_padding_mask=key_padding_mask)
        attn_output = self.dropout(attn_output)
        x_norm = self.norm1(x + attn_output)

        # Feed-forward network
        ffn_output = self.feed_forward(x_norm)
        ffn_output = self.dropout(ffn_output)
        output = self.norm2(x_norm + ffn_output)

        return output

In [4]:
# TransformerDecoderBlock

class TransformerDecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout, ffn_hidden_dim):
        super().__init__()

        # Multi-head self-attention
        self.self_attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(embed_dim, num_heads, dropout)

        self.dropout = nn.Dropout(dropout)

        # Layer norms
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)

        # Feed-forward network
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ffn_hidden_dim),
            nn.ReLU(),
            nn.Linear(ffn_hidden_dim, embed_dim))

    def forward(self, target, memory, memory_mask=None, memory_key_padding_mask=None,
                target_attn_mask=None, target_key_padding_mask=None):

        # Masked self-attention
        attn1, _ = self.self_attn(target, target, target, attn_mask=target_attn_mask,
                                  key_padding_mask=target_key_padding_mask)
        output = self.norm1(target + self.dropout(attn1))

        # Cross-attention
        attn2, _ = self.cross_attn(output, memory, memory, attn_mask=memory_mask,
                                   key_padding_mask=memory_key_padding_mask)
        output = self.norm2(output + self.dropout(attn2))

        # Feed-forward
        ffn = self.feed_forward(output)
        output = self.norm3(output + self.dropout(ffn))

        return output